# Session 9 · Search a support handbook by meaning

We will build a support assistant in four steps: **Load → Split → Store → Ask**. The input is one fictional company's 17-page Word handbook; the output is an answer supported by citations to that handbook.

| Step | What happens | Main result |
| --- | --- | --- |
| Load | Read paragraphs and tables in their original order. | One markdown string. |
| Split | Divide that string at section and topic headings. | Smaller documents with heading metadata. |
| Store | Ask Gemini to encode each piece and save it in Pinecone. | Searchable vectors, text, and metadata. |
| Ask | Find the top three matching pieces and give them to Gemini. | An answer with section citations. |

Gemini supplies **both** the embedding model and the chat model. Pinecone stores vectors and finds similar ones; it does not write the answer.

Before starting, follow the README to create and activate a **Python 3.13 virtual environment**, install `requirements.txt`, and copy `.env.example` to `.env`. Add your two keys, launch Jupyter from the repository folder, and select that environment's Python kernel. Run cells in order: later cells reuse variables created earlier, and a kernel restart clears those variables.

The notebook is saved without outputs. Loading and splitting are local operations; embedding, Pinecone operations, retrieval, and answer generation need network access and the corresponding API keys.

## 1 · Setup

These imports give Python the tools needed later. `csv` reads the evaluation file, `os` reads environment variables, `time` supplies polling pauses, `Path` builds file paths, NumPy performs vector arithmetic, and `python-docx` reads the Word file. Imports make tools available; they do not ingest the handbook or call a model.

The constants keep important choices visible in one place:

| Constant | Meaning |
| --- | --- |
| `EMBEDDING_MODEL` | `google_genai:` selects Google's LangChain integration; `gemini-embedding-001` selects the encoder. |
| `CHAT_MODEL` | Uses the same provider with `gemini-3.7-flash` to generate answers. |
| `HANDBOOK_PATH` | Relative path to the unchanged Word document; Jupyter should start in this repo. |
| `INDEX_NAME` | Name of our dedicated collection of vectors in Pinecone. |
| `K = 3` | Maximum number of matching passages retrieved per question. Increasing it sends more context to the answer model. |

A virtual environment controls which packages Python can import. The `.env` file is different: it supplies configuration values such as API keys.

In [ ]:
import csv  # Parse evaluation questions from CSV rows.
import os  # Read API keys from environment variables.
import time  # Pause while Pinecone starts or exposes new records.
from pathlib import Path  # Build file paths without OS-specific separators.
import numpy as np  # Use NumPy arrays and vector mathematics.
from docx import Document  # Open a Word document with python-docx.
from docx.table import Table  # Identify table objects while walking the document.
from dotenv import load_dotenv  # Load key-value pairs from the local .env file.

EMBEDDING_MODEL = "google_genai:gemini-embedding-001"  # Choose the Gemini embedding provider and model.
CHAT_MODEL = "google_genai:gemini-3.7-flash"  # Choose the Gemini answer provider and model.
HANDBOOK_PATH = Path("data/support_handbook.docx")  # Locate the unchanged handbook relative to this notebook.
INDEX_NAME = "trailmix-handbook"  # Name the Pinecone index used throughout the lab.
K = 3  # Retrieve three passages for each question.

### Load configuration without displaying secrets

`load_dotenv(Path(".env"))` reads `KEY=value` entries from the repository's `.env` file into the current process environment. By default it preserves values already set in the environment. It does not create the file, request keys, or verify that a key works.

`os.getenv(key)` returns a key's value, or `None` if it is absent. The list comprehension keeps only missing or empty keys; `if missing` tests whether that list contains any names. `", ".join(missing)` turns those names into a readable message without showing their values.

`GOOGLE_API_KEY` powers both Gemini clients; `PINECONE_API_KEY` authorizes vector database operations. If the cell reports a missing key, fill it in and rerun setup before continuing. The printed success message confirms presence, not successful authentication.

In [ ]:
load_dotenv(Path(".env"))  # Load .env values; existing environment values take priority.
missing = [key for key in ("GOOGLE_API_KEY", "PINECONE_API_KEY")  # Collect the names of any required keys that are missing.
           if not os.getenv(key)]  # Treat an unset or empty key as missing.
if missing:  # Run this branch if the missing-key list is not empty.
    print("Missing " + ", ".join(missing) + ". Add them to .env, then rerun.")  # Explain how to fix setup without revealing key values.
else:  # Use this branch when both keys are present.
    print("Both keys are present. Ready to build the lab!")  # Confirm presence only; this does not test key validity.

### Import the retrieval and answer building blocks

These imports connect the local document work to the services we will use. The factories `init_embeddings` and `init_chat_model` read a `provider:model` identifier so we can keep model names in constants.

| Tool | Role in this lab |
| --- | --- |
| `MarkdownHeaderTextSplitter` | Creates documents with `page_content` and heading `metadata`. |
| `PineconeVectorStore` | Embeds documents for storage and questions for search, using the supplied embedding client. |
| `ChatPromptTemplate` | Fills named placeholders in system and human messages. |
| `StrOutputParser` | Converts the chat response into ordinary text. |
| `RunnablePassthrough` | Carries the original question through the parallel input branches unchanged. |
| `Pinecone` / `ServerlessSpec` | Manage the index and choose its cloud and region. |

The `|` operator connects runnable steps, as in Session 8. Importing these classes alone does not run the pipeline.

In [ ]:
from langchain.embeddings import init_embeddings  # Create an embedding client from a provider:model string.
from langchain.chat_models import init_chat_model  # Create a chat client from a provider:model string.
from langchain_text_splitters import MarkdownHeaderTextSplitter  # Split markdown and retain headings as metadata.
from langchain_pinecone import PineconeVectorStore  # Connect LangChain documents to a Pinecone index.
from langchain_core.prompts import ChatPromptTemplate  # Build system and human messages with placeholders.
from langchain_core.output_parsers import StrOutputParser  # Extract plain text from the chat response.
from langchain_core.runnables import RunnablePassthrough  # Forward the original question through the chain.
from pinecone import Pinecone, ServerlessSpec  # Use the Pinecone client and serverless index settings.

## 2 · Load the Word document

`docx_to_markdown(path)` converts the file into one string. Its parameter `path` accepts either a filename string or a `Path`; the `-> str` annotation tells readers that the return value is text. These annotations describe expected types rather than performing conversion themselves.

`Document(path)` opens the Word file. `iter_inner_content()` walks top-level body paragraphs and tables in reading order, which matters when a table supplies facts for the paragraph above it. `isinstance(item, Table)` selects the table branch; its nested expression joins each row's cell text with ` | ` and joins the rows with `\n`.

For paragraphs, `.strip()` removes surrounding whitespace and skips empty text. The dictionary maps Word's `Heading 1` style to `# ` and `Heading 2` to `## `; `.get(item.style.name, "")` uses an empty prefix for other styles. Finally, `"\n\n".join(blocks)` puts blank lines between the converted blocks.

This is a text conversion, so fonts, page layout, headers, and footers are not part of the resulting search text. The original `.docx` is only read and remains unchanged.

In [ ]:
def docx_to_markdown(path: str | Path) -> str:  # Accept a path string or Path and return markdown text.
    """Convert body paragraphs and tables to markdown in reading order."""  # Describe the function for help() and readers.
    document = Document(path)  # Open the Word file at the supplied path.
    blocks = []  # Collect converted text blocks in reading order.
    for item in document.iter_inner_content():  # Visit body paragraphs and tables in their original order.
        if isinstance(item, Table):  # Handle a table separately from a paragraph.
            # Preserve each table's position, with one readable line per row.
            blocks.append("\n".join(" | ".join(c.text for c in r.cells) for r in item.rows))  # Join cells with pipes and rows with newlines.
        elif item.text.strip():  # Handle only paragraphs containing non-whitespace text.
            # Word styles become the headings that will supply citations.
            prefix = {"Heading 1": "# ", "Heading 2": "## "}.get(item.style.name, "")  # Map heading styles; use no prefix for other styles.
            blocks.append(prefix + item.text.strip())  # Add the heading marker and trimmed paragraph text.
    return "\n\n".join(blocks)  # Separate blocks with a blank line and return one string.

### Inspect what we loaded

We open the document once to count its top-level paragraphs and tables, and call our loader to produce the markdown used by the rest of the lab. `len(markdown)` counts characters, not words or model tokens. In an f-string, `:,` adds thousands separators to a number.

`markdown[:600]` is a slice: it shows the first 600 characters, starting at position zero. This preview lets you check the title and early text without flooding the notebook with the entire handbook. You should see **173 paragraphs, 8 tables, and 30,605 characters** with the supplied file and loader.

**17 pages is too long to read for every customer question — this is why we search.** We will store smaller passages and send only a few relevant ones to the answer model.

In [ ]:
document = Document(HANDBOOK_PATH)  # Open the document so we can count its body elements.
markdown = docx_to_markdown(HANDBOOK_PATH)  # Convert the same file to searchable markdown text.
print(f"Paragraphs: {len(document.paragraphs)}")  # Count top-level body paragraphs, including empty ones.
print(f"Tables: {len(document.tables)}")  # Count top-level tables, excluding any nested tables.
print(f"Total characters: {len(markdown):,}")  # Count characters; :, adds thousands separators.
print("\nFirst 600 characters:\n")  # Print a label with blank lines for readability.
print(markdown[:600])  # Show characters 0 through 599 without printing everything.

## 3 · What is an embedding?

An embedding represents text as a vector: a list of numeric coordinates that a model learns to place near vectors with related meanings. An individual coordinate is not a readable category such as “tent” or “waterproof”; the whole vector carries the representation.

`init_embeddings(EMBEDDING_MODEL)` initializes the Google embedding client. `embed_query(text)` takes a single query string and makes a Gemini API request to return its vector; this is the first live model request in the notebook. Later, `embed_documents(texts)` handles a list of passages for storage.

`len(vector)` measures the actual number of coordinates, which we will use for the Pinecone index dimension rather than assuming a fixed size. `vector[:5]` selects five values, and `round(value, 6)` changes only their displayed precision, not the vector we store. This cell needs a valid Google key and available quota.

In [ ]:
emb = init_embeddings(EMBEDDING_MODEL)  # Initialize the Gemini embedding client using our constant.
vector = emb.embed_query("My tent let water in during a storm.")  # Send one search-style text to Gemini and get its vector.
print(f"Vector length: {len(vector)}")  # Count the numeric coordinates in the returned vector.
print("First 5 numbers:", [round(value, 6) for value in vector[:5]])  # Show five coordinates, rounded to six decimal places.

## 4 · Similarity by meaning

Cosine similarity compares the directions of two nonzero vectors. In `cosine(a, b)`, `a` and `b` are embeddings; `np.asarray` turns their Python lists into NumPy arrays, `np.dot(a, b)` computes the dot product, and `np.linalg.norm` measures each vector's length. Dividing by both lengths removes magnitude from the comparison; larger cosine values indicate a closer directional match.

The first text is our query-like reference, followed by a tent fact and a socks fact. `embed_documents(sentences)` embeds all three with the same document task setting for this demonstration. During actual retrieval, the vector store uses document embeddings for stored passages and query embeddings for questions.

`vectors[0]` selects the reference vector. `[1:]` skips the first item, and `zip` pairs each remaining sentence with its corresponding vector. `:.3f` displays three decimal places. The tent fact should score higher even though it shares few exact words with “my tent let water in”; inspect the actual scores rather than treating this expectation as a guaranteed result.

In [ ]:
def cosine(a, b):  # Define a helper that compares two nonzero vectors.
    a, b = np.asarray(a), np.asarray(b)  # Convert both Python lists into NumPy arrays.
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))  # Divide their dot product by their lengths; return a float.

sentences = [  # Collect the query-like text and two candidate passages.
    "my tent let water in",  # Use this first sentence as the comparison reference.
    "The rain fly has a 3,000 mm waterproof rating.",  # Use this tent-related sentence as one candidate.
    "Trail Socks are merino-wool hiking socks sold in packs of three.",  # Use this unrelated socks sentence as the other candidate.
]  # Finish the ordered list of three texts.
vectors = emb.embed_documents(sentences)  # Embed all three texts with the same document task setting.
for sentence, candidate in zip(sentences[1:], vectors[1:]):  # Pair each candidate sentence with its corresponding vector.
    print(f"{cosine(vectors[0], candidate):.3f}  {sentence}")  # Compare each candidate to the first vector; show 3 decimals.

## 5 · Split by heading

A piece should contain enough information to answer a focused question while remaining small enough to retrieve selectively. Here we use the handbook's own structure instead of a fixed character count.

`headers_to_split_on` is a list of `(markdown marker, metadata key)` pairs. `("#", "section")` stores a top-level heading such as `Policies` under `metadata["section"]`; `("##", "topic")` stores a subheading such as `Returns` under `metadata["topic"]`. The splitter removes these heading lines from the passage text by default and retains their names in metadata.

`split_text(markdown)` returns a list of LangChain `Document` objects, each with `.page_content` for text and `.metadata` for labels. With this handbook, expect **59 pieces**: one front-matter piece and 58 topic pieces. Seven section headings do not add seven extra pieces because they introduce topics without separate body text before the first topic.

In [ ]:
splitter = MarkdownHeaderTextSplitter(  # Configure a splitter that understands markdown headings.
    headers_to_split_on=[("#", "section"), ("##", "topic")]  # Store # titles as section and ## titles as topic metadata.
)  # Finish configuring the splitter.
pieces = splitter.split_text(markdown)  # Convert the markdown string into LangChain Documents.
print(f"Pieces: {len(pieces)}")  # Count the resulting searchable pieces.
print("Expected source outline: 7 sections and 58 topics.")  # Show the source outline to compare with the piece count.

### Turn metadata into citation labels

The parameter `piece` is one LangChain document. `piece.metadata.get("section", "Front matter")` returns its section or a default label when there is no heading; `.get("topic")` returns the topic or `None` if absent.

The conditional expression returns `[section > topic]` when a topic exists and `[section]` otherwise. The title-page piece has neither field, so it becomes `[Front matter]`. This helper formats existing heading labels; it does not invent sources or assign page numbers.

`pieces[:15]` limits the preview to the first 15 documents. Each line shows the citation alongside `len(piece.page_content)`, helping you inspect the split before sending anything to Pinecone.

In [ ]:
def citation_label(piece):  # Define how a document becomes a citation label.
    section = piece.metadata.get("section", "Front matter")  # Read its section, defaulting to Front matter if absent.
    topic = piece.metadata.get("topic")  # Read its topic; get() returns None when it is absent.
    return f"[{section} > {topic}]" if topic else f"[{section}]"  # Include the topic only when one exists.

for piece in pieces[:15]:  # Inspect only the first 15 pieces.
    print(f"{citation_label(piece)} — {len(piece.page_content):,} characters")  # Show each citation and its text length with separators.

## 6 · Create the Pinecone index

An index is configured for vectors with one dimension and one similarity metric. We measure `len(emb.embed_query("test"))` so its dimension matches the selected encoder; the word “test” only supplies a short sample input for that measurement.

`Pinecone(api_key=...)` creates the authenticated client. `has_index(INDEX_NAME)` checks whether the named index already exists; `if not` requests creation only when it is absent. Existing indexes are checked for compatibility in the next cell.

| Parameter | What it controls |
| --- | --- |
| `name=INDEX_NAME` | The index name used again for storage and retrieval. |
| `dimension=dimension` | Number of numeric coordinates required for each stored vector. |
| `metric="cosine"` | Compare vector directions; higher search scores indicate greater similarity. |
| `spec=ServerlessSpec(...)` | Choose a serverless deployment. |
| `cloud="aws"` | Host the index on Amazon Web Services. |
| `region="us-east-1"` | Choose the AWS region for this lab's index. |

Index creation is a live Pinecone operation. Printing “Created” means the creation request completed, so we must still wait for readiness.

In [ ]:
dimension = len(emb.embed_query("test"))  # Measure a real embedding rather than guessing its length.
print(f"Embedding dimension: {dimension}")  # Display the dimension required by this embedding model.
pc = Pinecone(api_key=os.environ["PINECONE_API_KEY"])  # Authenticate the Pinecone client using the stored key.
if not pc.has_index(INDEX_NAME):  # Create an index only when this name is absent.
    pc.create_index(  # Send a request to create the index.
        name=INDEX_NAME, dimension=dimension, metric="cosine",  # Set its name, vector size, and similarity metric.
        spec=ServerlessSpec(cloud="aws", region="us-east-1"),  # Choose serverless hosting on AWS in this region.
    )  # Finish the index-creation request.
    print(f"Created {INDEX_NAME}.")  # Report that creation was requested, not yet completed.
else:  # Use this branch if an index already has this name.
    print(f"Reusing {INDEX_NAME}.")  # Report that we will reuse the existing index.

### Wait for readiness and check compatibility

Creating an index is asynchronous: the service may accept the request before the index is ready to use. `describe_index` returns its current settings and status. The assignment expression `description := ...` saves that response while the loop tests its `status["ready"]` value.

`time.monotonic() + 120` creates a deadline two minutes in the future using a clock suited to elapsed-time checks. `time.sleep(1)` waits one second between polls; the timeout prevents an endless loop if provisioning stalls.

An existing index must match both `dimension` and `metric`. If either differs, the cell raises a readable error before any upsert; for this dedicated lab index, recreate it and ingest again, remembering that deletion removes its stored vectors. `pc.Index(host=description.host)` then creates a connection to the ready index's data endpoint; it does not create another index.

In [ ]:
deadline = time.monotonic() + 120  # Set a deadline 120 seconds from the monotonic clock.
while not (description := pc.describe_index(INDEX_NAME)).status["ready"]:  # Fetch status and keep polling while ready is False.
    if time.monotonic() >= deadline:  # Stop polling if the two-minute deadline has passed.
        raise TimeoutError("Index is not ready; wait and retry this cell.")  # Raise a readable error instead of waiting forever.
    time.sleep(1)  # Pause for one second between status requests.
if description.dimension != dimension or description.metric != "cosine":  # Reject an existing index with incompatible settings.
    raise ValueError("Index mismatch: delete and recreate the lab index, then rerun.")  # Explain that the dedicated lab index needs recreation.
index = pc.Index(host=description.host)  # Connect directly to the ready index using its host.
print("Index ready.")  # Confirm the index is ready to accept data.

## 7 · Store

`PineconeVectorStore(index=index, embedding=emb)` binds the ready database index to the Gemini encoder. The `index` parameter selects where records go; `embedding` selects how text is converted to vectors for both writes and searches.

The list comprehension creates IDs `piece-0`, `piece-1`, and so on from each piece's position. In `add_documents(pieces, ids=ids)`, `pieces` supplies the text and heading metadata, and `ids` supplies one identifier per piece. The integration embeds each passage and upserts its vector, text, and metadata; writing the same ID again replaces that record rather than adding a duplicate.

The five-second sleep allows time for writes to become searchable, but visibility can take longer. If the next search is empty, wait and retry it. Stable IDs work for rerunning this unchanged handbook; changing the piece count or splitting rules can leave old IDs behind, so use a fresh dedicated index for such changes.

In [ ]:
store = PineconeVectorStore(index=index, embedding=emb)  # Wrap the ready index and Gemini encoder as a vector store.
ids = [f"piece-{i}" for i in range(len(pieces))]  # Assign repeatable IDs from piece-0 to the final piece.
store.add_documents(pieces, ids=ids)  # Embed and upsert text plus its heading metadata.
time.sleep(5)  # Wait five seconds for new records to become searchable.
print(f"Stored {len(pieces)} pieces with stable IDs.")  # Report how many pieces were sent for storage.

## 8 · Search

`similarity_search_with_score(query, k=K)` converts the question into a query embedding, searches Pinecone, and returns up to `K` document-and-score pairs. Here `K` is three. This step retrieves evidence; it does not call the chat model or generate an answer.

The loop unpacks each pair into `piece` and `score`. We print the heading citation, the score rounded for display with `:.3f`, and a preview built from the first 120 characters. `.replace("\n", " ")` flattens line breaks in the preview without changing stored text.

With this cosine index, higher scores indicate a closer match, not a probability that a passage is correct. Look for `[Support teams > Safety escalations]` and inspect the retrieved text before relying on an answer.

In [ ]:
results = store.similarity_search_with_score(  # Search for similar passages and return their scores.
    "customer smells gas from their stove", k=K  # Embed this question and request the top K matches.
)  # Finish the search request.
for piece, score in results:  # Unpack each result into its document and numeric score.
    preview = piece.page_content[:120].replace("\n", " ")  # Take 120 characters and flatten newlines for a short preview.
    print(f"{citation_label(piece)} | score {score:.3f}\n{preview}\n")  # Print the citation, score to 3 decimals, and preview.

## 9 · Evaluate retrieval

This checks whether search finds the intended handbook topic for five paraphrased questions. `Path(...).open(newline="", encoding="utf-8")` reads the CSV as UTF-8 while leaving newline handling to the `csv` module. `csv.DictReader` uses the header row as dictionary keys: `question`, `expected_section`, and `expected_topic`.

Each `similarity_search(..., k=K)` returns documents without scores or a generated answer. `any(...)` marks a hit when **one same retrieved document** matches both expected metadata fields; finding the section in one document and the topic in another does not count. `int(hit)` converts `True` to 1 and `False` to 0 for the counter.

**Hit rate @3** means “how many evaluation questions had the expected topic among the top three results?” The target is at least **4/5**; `5/5` means all five questions retrieved their target. This measures retrieval coverage, not answer correctness or citation quality, which we inspect separately. Expected and retrieved labels are printed so you can diagnose a miss.

In [ ]:
with Path("data/eval_questions.csv").open(newline="", encoding="utf-8") as file:  # Open the CSV as UTF-8; newline="" lets csv handle line endings.
    questions = list(csv.DictReader(file))  # Read rows into dictionaries keyed by column names.
hits = 0  # Start the successful-retrieval counter at zero.
for row in questions:  # Evaluate each question independently.
    found = store.similarity_search(row["question"], k=K)  # Retrieve the top K documents without generating an answer.
    hit = any(p.metadata.get("section") == row["expected_section"]  # Check that a retrieved document has the expected section.
              and p.metadata.get("topic") == row["expected_topic"] for p in found)  # Also require its expected topic on that same document.
    hits += int(hit)  # Add 1 for a hit or 0 for a miss.
    print(f"{'✓' if hit else '✗'} {row['question']}")  # Print a check or cross beside the question.
    print(f"  Expected: [{row['expected_section']} > {row['expected_topic']}]")  # Show the target citation from the evaluation row.
    print("  Found:", ", ".join(citation_label(p) for p in found))  # Show the retrieved citations so misses can be inspected.
print(f"\nHit rate @3: {hits}/{len(questions)}")  # Print successful questions over total questions at rank 3.

## 10 · Ask with citations

`format_docs(pieces)` accepts the list of retrieved documents and returns one prompt-ready string. For each piece, it places the exact citation label on its own line above the passage; `"\n\n".join(...)` separates passages with blank lines so the answer model can see which label belongs to which text.

`store.as_retriever(search_kwargs={"k": K})` wraps the vector store as a LangChain runnable. The `search_kwargs` dictionary forwards settings to search; its `k` entry requests the top three passages. Calling the retriever with a question returns matching documents, while `format_docs` converts them into the `excerpts` string needed by our prompt.

In [ ]:
def format_docs(pieces):  # Define how retrieved Documents become prompt text.
    return "\n\n".join(f"{citation_label(p)}\n{p.page_content}" for p in pieces)  # Prefix each passage with its citation and separate passages.

retriever = store.as_retriever(search_kwargs={"k": K})  # Return at most K documents whenever the retriever is invoked.

### Define what the answer model sees

The adjacent strings inside `system_message` form one instruction string. It tells Gemini to use only the retrieved excerpts, copy the bracketed heading labels after each claim, and use a fixed fallback when those excerpts do not cover the question.

`ChatPromptTemplate.from_messages` takes an ordered list of `(role, content)` tuples. `"system"` supplies the general answer instructions; `"human"` supplies this request's evidence and question. `{excerpts}` and `{question}` are template placeholders filled by the chain, while `\n` inserts a line break into the resulting message.

`init_chat_model(CHAT_MODEL)` initializes the Google chat client; the generation request happens when we invoke the finished chain. Prompt instructions guide behavior but do not verify it: inspect actual answers for unsupported claims, missing citations, or an incorrect fallback. “The excerpts do not cover this” can also result from retrieval missing a relevant passage, so retrieval quality matters.

In [ ]:
system_message = (  # Assemble one system message from adjacent string literals.
    "You are a TrailMix support assistant. Answer only from the handbook excerpts below. "  # Limit answers to the handbook excerpts provided below.
    "After each claim, cite the section in square brackets exactly as shown, "  # Require exact square-bracket citations after each claim.
    "e.g. [Policies > Returns]. If the excerpts do not answer the question, say: "  # Give a citation example and introduce the fallback rule.
    "'The handbook doesn't cover this — route to General Support.'"  # Specify the required reply when excerpts lack the answer.
)  # Finish the system instruction string.
prompt = ChatPromptTemplate.from_messages([  # Build a reusable template with two message roles.
    ("system", system_message),  # Place the instruction string in the system role.
    ("human", "Handbook excerpts:\n{excerpts}\n\nQuestion: {question}"),  # Fill these named placeholders when the chain runs.
])  # Finish the prompt template.
model = init_chat_model(CHAT_MODEL)  # Initialize the Gemini chat client for writing answers.

### Connect the Session 8 runnable chain

The chain receives one input string: the customer's question. Its first dictionary creates two named outputs from that same input. In the `excerpts` branch, `retriever | format_docs` finds documents and formats their cited text; in the `question` branch, `RunnablePassthrough()` forwards the question unchanged. LangChain treats this dictionary as parallel runnable branches.

The prompt consumes that dictionary because its keys match `{excerpts}` and `{question}`. The next `|` passes the filled messages to Gemini, and `StrOutputParser()` extracts response text from the model message. The parser does not validate citations or check facts.

Building `chain` defines the workflow; it does not run retrieval or generate an answer yet. `chain.invoke(question)` starts the workflow for one question, including a query-embedding request, a Pinecone search, and a Gemini answer request.

In [ ]:
chain = (  # Compose a runnable chain; construction does not answer yet.
    {"excerpts": retriever | format_docs, "question": RunnablePassthrough()}  # Build excerpts by retrieval, and preserve the question text.
    | prompt | model | StrOutputParser()  # Fill the prompt, call Gemini, then extract answer text.
)  # Finish defining the reusable chain.

### Inspect complete answers

The list contains two questions that the handbook should cover and one about the CEO's favourite hiking trail that it does not cover. The `for` loop calls `chain.invoke(question)` once per item, so each question triggers its own retrieval and answer request. The f-strings print the question before the returned answer with a blank line between examples.

For the gas question, look for the stop-use and Safety Team guidance with `[Support teams > Safety escalations]`; for worn boots, look for the footwear exception in `[Policies > Returns]`. Compare every factual claim with the actual cited text rather than accepting a plausible label as proof.

For the CEO question, expect “The handbook doesn't cover this — route to General Support.” If Gemini invents an answer or attaches a citation to an unsupported claim, the lab has revealed a generation failure even when retrieval hit rate was high.

In [ ]:
demo_questions = [  # Collect two handbook questions and one uncovered question.
    "Customer says they can smell gas from their stove. What do I do?",  # Test safety escalation instructions.
    "Can someone send back boots they already wore on one hike?",  # Test the return policy and footwear exception.
    "What is the CEO's favourite hiking trail?",  # Test whether the model uses the uncovered-question fallback.
]  # Finish the list of demonstration questions.
for question in demo_questions:  # Run the complete chain separately for each question.
    print(f"Question: {question}")  # Print the input question before its answer.
    print(f"Answer: {chain.invoke(question)}\n")  # Retrieve passages, call Gemini, and print its answer.

## 11 · Your turn

Replace the string assigned to `question`, then rerun this cell. `chain.invoke(question)` uses the same stored handbook and retriever settings; you do not need to rerun ingestion for every new question. A fresh kernel does require rerunning the setup and chain-building cells to restore the Python variables.

Try these questions:

- “Can you ship my order to Japan?” — look for `[Policies > International shipping]`.
- “What do I get with TrailMix Club?” — look for `[Policies > TrailMix Club membership]`.
- “My Beacon 400 headlamp will not turn on; what should I try?” — look for `[Products > Beacon 400 Headlamp]`.

Compare the question, retrieved passages, and answer. If you change `K` to experiment with context size, rerun the retriever and chain cells before asking again; the existing retriever already holds its earlier setting.

In [ ]:
question = "Can you ship my order to Japan?"  # Replace this string with your own customer question.
print(chain.invoke(question))  # Run retrieval and generation, then display the cited answer.